# Анализ результатов MGPU-VoxelWaterfall

Этот notebook читает только явно заданный `PUBLICATION_ROOT`. Синтетические результаты не создаются.


In [ ]:
from pathlib import Path
import csv
import json
from IPython.display import Markdown, SVG, display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name != "MGPU-VoxelWaterfall":
    candidate = PROJECT_ROOT / "MGPU-VoxelWaterfall"
    if candidate.exists():
        PROJECT_ROOT = candidate
PUBLICATION_ROOT = PROJECT_ROOT / "Research" / "artifacts" / "publication"
print(f"PUBLICATION_ROOT={PUBLICATION_ROOT}")


## Загрузка артефактов

Если strict PASS `analysis_summary.v2.json` отсутствует, выводится `NOT_MEASURED`.


In [ ]:
def read_json(path: Path):
    if not path.exists():
        return None
    return json.loads(path.read_text(encoding="utf-8"))

def read_csv_rows(path: Path):
    if not path.exists():
        return []
    with path.open(encoding="utf-8", newline="") as handle:
        return list(csv.DictReader(handle))

def candidate_summary_dirs(root: Path):
    return [root, root / "Smoke", root / "Full", root / "smoke", root / "full"]

def load_pass_summaries(root: Path):
    loaded = []
    seen = set()
    for directory in candidate_summary_dirs(root):
        summary_path = directory / "analysis_summary.v2.json"
        if summary_path in seen or not summary_path.exists():
            continue
        seen.add(summary_path)
        data = read_json(summary_path)
        if isinstance(data, dict):
            loaded.append({"path": str(summary_path), "suite": data.get("suite", directory.name), "status": data.get("status", "NOT_MEASURED"), "summary": data if data.get("status") == "PASS" else None})
    return loaded

summaries = load_pass_summaries(PUBLICATION_ROOT)
if not summaries:
    display(Markdown("**NOT_MEASURED:** strict PASS `analysis_summary.v2.json` not found."))
else:
    rows = [f"| {row['suite']} | {row['status']} | `{row['path']}` |" for row in summaries]
    table = """| suite | status | path |
|---|---|---|
""" + "\n".join(rows)
    display(Markdown(table))


## Гипотезы

Читаются текущие ключи schema v2: `mean_difference`, `median_difference`, `difference_ci95`, `effect_direction`.


In [ ]:
def hypothesis_row(suite, name, result):
    if not isinstance(result, dict):
        return {"suite": suite, "hypothesis": name, "result": "NOT_MEASURED", "paired_n": None, "mean_difference": None, "median_difference": None, "ci_status": "NOT_MEASURED", "ci_lower": None, "ci_upper": None, "effect_direction": "NOT_MEASURED"}
    stats = result.get("statistics") or {}
    ci = stats.get("difference_ci95") or stats.get("reduction_ci95") or {}
    return {"suite": suite, "hypothesis": name, "result": result.get("result", "NOT_MEASURED"), "paired_n": stats.get("paired_n", result.get("paired_n")), "mean_difference": stats.get("mean_difference", stats.get("mean_reduction")), "median_difference": stats.get("median_difference", stats.get("median_reduction")), "ci_status": ci.get("status", "NOT_MEASURED") if isinstance(ci, dict) else "NOT_MEASURED", "ci_lower": ci.get("lower") if isinstance(ci, dict) else None, "ci_upper": ci.get("upper") if isinstance(ci, dict) else None, "effect_direction": result.get("effect_direction", "NOT_MEASURED")}

hypothesis_rows = []
for item in summaries:
    data = item.get("summary")
    suite = item.get("suite")
    hypotheses = data.get("hypothesis_results", {}) if isinstance(data, dict) else {}
    for name in ["H1", "H2", "H3", "RQ3"]:
        hypothesis_rows.append(hypothesis_row(suite, name, hypotheses.get(name)))

if not hypothesis_rows:
    display(Markdown("**NOT_MEASURED**"))
else:
    header = """| suite | hypothesis | result | n | mean | median | CI | direction |
|---|---|---:|---:|---:|---:|---|---|"""
    lines = []
    for row in hypothesis_rows:
        ci_text = row["ci_status"] if row["ci_status"] != "MEASURED" else f"[{row['ci_lower']}, {row['ci_upper']}]"
        lines.append(f"| {row['suite']} | {row['hypothesis']} | {row['result']} | {row['paired_n']} | {row['mean_difference']} | {row['median_difference']} | {ci_text} | {row['effect_direction']} |")
    display(Markdown(header + "\n" + "\n".join(lines)))


## График H1

График строится только для measured CI. Подписи на графике на английском.


In [ ]:
def h1_svg(rows):
    measured = [row for row in rows if row["hypothesis"] == "H1" and row["ci_status"] == "MEASURED"]
    if not measured:
        return None
    low = min(0.0, *(float(row["ci_lower"]) for row in measured))
    high = max(0.0, *(float(row["ci_upper"]) for row in measured))
    span = high - low if high > low else 1.0
    width = 760
    left = 180
    row_h = 40
    height = 70 + row_h * len(measured)
    def x(value):
        return left + (float(value) - low) / span * (width - left - 40)
    parts = [f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">', '<rect width="100%" height="100%" fill="white"/>', '<text x="20" y="28" font-family="Segoe UI,Arial" font-size="18" font-weight="600">H1 Paired End-to-End Difference CI</text>', f'<line x1="{x(0.0):.1f}" y1="48" x2="{x(0.0):.1f}" y2="{height - 20}" stroke="#666" stroke-dasharray="4 4"/>']
    for index, row in enumerate(measured):
        y = 70 + index * row_h
        parts.append(f'<text x="20" y="{y + 5}" font-family="Segoe UI,Arial" font-size="13">{row["suite"]}</text>')
        parts.append(f'<line x1="{x(row["ci_lower"]):.1f}" y1="{y}" x2="{x(row["ci_upper"]):.1f}" y2="{y}" stroke="#1f5f99" stroke-width="2"/>')
        parts.append(f'<circle cx="{x(row["mean_difference"]):.1f}" cy="{y}" r="4" fill="#1f5f99"/>')
    parts.append('</svg>')
    return "\n".join(parts)

svg = h1_svg(hypothesis_rows)
if svg is None:
    display(Markdown("**NOT_MEASURED:** measured H1 CI is absent."))
else:
    display(SVG(svg))


## CSV артефакты

Читаются только CSV внутри `PUBLICATION_ROOT`.


In [ ]:
csv_names = ["raw_frames.csv", "runs.csv", "paired_runs.csv", "invalid_records.csv"]
csv_rows = []
for directory in candidate_summary_dirs(PUBLICATION_ROOT):
    for name in csv_names:
        path = directory / name
        if path.exists():
            csv_rows.append({"directory": str(directory), "file": name, "rows": len(read_csv_rows(path))})
if not csv_rows:
    display(Markdown("**NOT_MEASURED:** publication CSV artifacts not found."))
else:
    lines = [f"| `{row['directory']}` | {row['file']} | {row['rows']} |" for row in csv_rows]
    table = """| directory | file | rows |
|---|---|---:|
""" + "\n".join(lines)
    display(Markdown(table))


## Вывод

Без strict PASS данных и measured CI поддержка H1/H2/H3 не утверждается.
